In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from datetime import datetime
from scipy import stats as sp_stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

In [ ]:
US_EQUITY  = ['VIOV', 'MTUM', 'VBR']
ETF_GLOBAL = ['VEA', 'VWO', 'AGG', 'BNDX', 'GLD', 'VNQ', 'BTC-USD']
STOCKS     = ['MC.PA', 'ASML', 'NESN.SW', 'SIE.DE']
BENCHMARK  = ['VT', 'AGG']
ALL_TICKERS = list(set(US_EQUITY + ETF_GLOBAL + STOCKS + BENCHMARK))
START = '2013-01-01'
END   = datetime.today().strftime('%Y-%m-%d')
raw = yf.download(ALL_TICKERS, start=START, end=END, auto_adjust=True, progress=True)['Close']
print(raw.shape)
print(raw.isna().sum().sort_values(ascending=False))

## 1. Investment Policy Statement (IPS)

In [ ]:
from IPython.display import Markdown, display

ips = '''
| Attribut | Détail |
|---|---|
| **Profil investisseur** | Individuel — Patrimoine privé |
| **Actif existant** | CHF 180 000 en AstraZeneca (Large Cap Healthcare) |
| **Horizon** | Long terme — 10 à 15 ans |
| **Tolérance au risque** | Modérée-élevée (drawdowns > 20% acceptés) |
| **Objectif de rendement** | Inflation + 4% net de frais, annualisé |
| **Besoins en revenus** | Faibles — stratégie de capitalisation totale |
| **Contraintes** | Long-only · Pas de levier · TER < 0.5%/an |
| **Tilts factoriels** | Small Cap · Value · High Momentum |
| **Devise de référence** | CHF (expositions USD/EUR non hedgées acceptées) |
| **Contrainte de concentration** | Max 40% par actif individuel |
'''
display(Markdown('### Profil Investisseur — Sofia\n' + ips))

## 2. Strategic Asset Allocation (SAA)

In [ ]:
PORTFOLIO_TICKERS = list(dict.fromkeys(US_EQUITY + ETF_GLOBAL + STOCKS))
prices  = raw[PORTFOLIO_TICKERS].ffill().dropna()
returns = np.log(prices / prices.shift(1)).dropna()
ann_ret = returns.mean() * 252
ann_vol = returns.std()  * np.sqrt(252)
RF_PROXY = 0.03
sharpe_i = (ann_ret - RF_PROXY) / ann_vol
stats = pd.DataFrame({
    'Rendement annualisé': ann_ret,
    'Volatilité annualisée': ann_vol,
    'Sharpe individuel': sharpe_i
}).sort_values('Sharpe individuel', ascending=False)
print(f'Période : {returns.index[0].date()} → {returns.index[-1].date()}')
print(f'Actifs  : {len(PORTFOLIO_TICKERS)}   |   Obs : {len(returns)}\n')
print(stats.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Statistiques individuelles annualisées', fontsize=14, fontweight='bold')
ax1 = axes[0]
colors_group = (['#d62728']*3 + ['#1f77b4']*7 + ['#2ca02c']*4)
for t, c in zip(PORTFOLIO_TICKERS, colors_group):
    ax1.scatter(ann_vol[t], ann_ret[t], s=90, color=c, zorder=3)
    ax1.annotate(t, (ann_vol[t], ann_ret[t]), textcoords='offset points', xytext=(5,3), fontsize=8)
ax1.axhline(0, color='grey', lw=0.8, linestyle='--')
ax1.set_xlabel('Volatilité annualisée'); ax1.set_ylabel('Rendement annualisé')
ax1.set_title('Espace Risque / Rendement')
ax2 = axes[1]
sharpe_sorted = stats['Sharpe individuel'].sort_values()
bar_colors = ['#d62728' if v < 0 else '#2ca02c' for v in sharpe_sorted]
sharpe_sorted.plot(kind='barh', ax=ax2, color=bar_colors, alpha=0.85)
ax2.axvline(0, color='grey', lw=0.8, linestyle='--')
ax2.set_title('Sharpe Ratio individuel (rf = 3%)')
plt.tight_layout(); plt.show()

In [ ]:
# Matrice de corrélation — rendements journaliers
corr = returns.corr()
fig, ax = plt.subplots(figsize=(13, 10))
im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(PORTFOLIO_TICKERS)))
ax.set_yticks(range(len(PORTFOLIO_TICKERS)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(corr.index, fontsize=8)
for i in range(len(corr)):
    for j in range(len(corr)):
        val = corr.values[i, j]
        col = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=col)
plt.colorbar(im, ax=ax, fraction=0.02, label='Corrélation')
ax.set_title('Matrice de Corrélation — Rendements journaliers', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()

# Diversificateurs : actifs les moins corrélés en moyenne
corr_mean = corr.mean().sort_values()
print('Corrélation moyenne par actif (meilleurs diversificateurs en tête) :')
print(corr_mean.round(3))

## 3. Instrument Selection

Chaque instrument doit disposer d'au moins **10 ans d'historique** pour supporter le backtest.

In [ ]:
history = pd.DataFrame({
    'Première date': prices.apply(lambda x: x.first_valid_index()),
    'Dernière date': prices.apply(lambda x: x.last_valid_index()),
})
history['Années'] = ((history['Dernière date'] - history['Première date']).dt.days / 365.25).round(1)
history['✓ 10 ans'] = history['Années'] >= 10
history = history.sort_values('Années', ascending=False)
print('=== Disponibilité historique par actif ===\n')
print(history.to_string())
print(f"\n{history['✓ 10 ans'].sum()}/{len(history)} actifs valident le critère 10 ans.")

fig, ax = plt.subplots(figsize=(12, 5))
colors_h = ['#2ca02c' if v else '#d62728' for v in history['✓ 10 ans']]
history['Années'].plot(kind='barh', ax=ax, color=colors_h, alpha=0.85)
ax.axvline(10, color='black', lw=1.5, linestyle='--', label='Seuil 10 ans')
ax.set_title('Historique disponible par actif', fontweight='bold')
ax.set_xlabel('Années de données')
ax.legend()
plt.tight_layout(); plt.show()

## 4. Portfolio Construction

### 4.1 Fixed-Mix — US Equity Sub-Portfolio & Portefeuille Global

In [ ]:
# --- US Equity Sub-Portfolio (equal-weight fixed-mix) ---
US_SUB = ['VIOV', 'MTUM', 'VBR']
w_us_fixed = pd.Series({t: 1/3 for t in US_SUB}, name='US Equity Fixed-Mix')

# --- Portefeuille global diversifié (fixed-mix) ---
GLOBAL_FIXED = pd.Series({
    'VIOV': 0.10, 'MTUM': 0.10, 'VBR':     0.10,
    'VEA':  0.12, 'VWO':  0.08,
    'AGG':  0.15, 'BNDX': 0.10,
    'GLD':  0.07, 'VNQ':  0.05,
    'MC.PA':0.05, 'ASML': 0.04, 'NESN.SW': 0.02, 'SIE.DE': 0.02,
}, name='Global Fixed-Mix')

print('US Equity Sub-Portfolio :'); print(w_us_fixed.to_string())
print(f'Total : {w_us_fixed.sum():.0%}\n')
print('Portefeuille Global Diversifié :'); print(GLOBAL_FIXED.to_string())
print(f'Total : {GLOBAL_FIXED.sum():.0%}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Allocation Fixed-Mix', fontsize=13, fontweight='bold')
w_us_fixed.plot(kind='bar', ax=axes[0], color='#d62728', alpha=0.85)
axes[0].set_title('US Equity Sub-Portfolio'); axes[0].tick_params(axis='x', rotation=0)
GLOBAL_FIXED.sort_values().plot(kind='barh', ax=axes[1], color='#1f77b4', alpha=0.85)
axes[1].set_title('Portefeuille Global Diversifié')
plt.tight_layout(); plt.show()

### 4.2 Optimisation des poids — FF3 & CAPM (Max Sharpe)

In [ ]:
try:
    import pandas_datareader.data as web
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas-datareader', '-q'])
    import pandas_datareader.data as web

ff3_raw = web.DataReader('F-F_Research_Data_Factors_daily', 'famafrench', start=START)[0]
if not isinstance(ff3_raw.index, pd.DatetimeIndex):
    ff3_raw.index = pd.to_datetime(ff3_raw.index.astype(str), format='%Y%m%d')
ff3 = ff3_raw / 100

mkt_ret_full = np.log(raw['VT'] / raw['VT'].shift(1)).dropna()
common_idx = (returns.index
              .intersection(ff3.index)
              .intersection(mkt_ret_full.index))
R = returns.loc[common_idx, PORTFOLIO_TICKERS]
F = ff3.loc[common_idx]

rf_daily  = F['RF']
rf_annual = rf_daily.mean() * 252
factors   = F[['Mkt-RF', 'SMB', 'HML']]

betas_ff3 = {}; alphas_ff3 = {}; r2_ff3 = {}
for t in PORTFOLIO_TICKERS:
    excess = (R[t] - rf_daily).values
    X = np.column_stack([np.ones(len(factors)), factors.values])
    coeffs, _, _, _ = np.linalg.lstsq(X, excess, rcond=None)
    y_pred = X @ coeffs
    ss_res = np.sum((excess - y_pred)**2)
    ss_tot = np.sum((excess - excess.mean())**2)
    alphas_ff3[t] = coeffs[0] * 252
    betas_ff3[t]  = dict(zip(['Mkt-RF','SMB','HML'], coeffs[1:]))
    r2_ff3[t]     = 1 - ss_res / ss_tot

betas_df      = pd.DataFrame(betas_ff3).T
factor_premia = factors.mean() * 252
mu_ff3 = pd.Series({
    t: rf_annual + betas_ff3[t]['Mkt-RF']*factor_premia['Mkt-RF']
       + betas_ff3[t]['SMB']*factor_premia['SMB'] + betas_ff3[t]['HML']*factor_premia['HML']
    for t in PORTFOLIO_TICKERS}, name='E[r] FF3')
print('Primes FF3 annualisées :'); print(factor_premia.round(4))
print(f'rf annualisé : {rf_annual:.4f}\n')
print('Rendements attendus FF3 :'); print(mu_ff3.sort_values(ascending=False).round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Fama-French 3 Facteurs — Bêtas par actif', fontsize=14, fontweight='bold')
factor_meta = [('Mkt-RF','#d62728','β Marché'),('SMB','#1f77b4','β Small Cap'),('HML','#2ca02c','β Value')]
for ax, (factor, color, label) in zip(axes, factor_meta):
    betas_df[factor].sort_values().plot(kind='barh', ax=ax, color=color, alpha=0.85)
    ax.axvline(0, color='black', lw=0.8); ax.set_title(label)
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(6, 8))
im = ax.imshow(betas_df.values, cmap='RdBu_r', aspect='auto', vmin=-0.5, vmax=1.5)
ax.set_xticks(range(3)); ax.set_xticklabels(['Mkt-RF','SMB','HML'])
ax.set_yticks(range(len(PORTFOLIO_TICKERS))); ax.set_yticklabels(betas_df.index)
for i in range(len(PORTFOLIO_TICKERS)):
    for j in range(3):
        ax.text(j, i, f'{betas_df.values[i,j]:.2f}', ha='center', va='center', fontsize=9, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.03)
ax.set_title('Heatmap bêtas FF3', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
mkt_ret     = np.log(raw['VT'] / raw['VT'].shift(1)).dropna()
mkt_aligned = mkt_ret.loc[common_idx]
betas_capm = {}; alphas_capm = {}; r2_capm = {}
for t in PORTFOLIO_TICKERS:
    excess_i   = R[t] - rf_daily
    excess_mkt = mkt_aligned - rf_daily
    slope, intercept, r_val, _, _ = sp_stats.linregress(excess_mkt, excess_i)
    betas_capm[t] = slope; alphas_capm[t] = intercept*252; r2_capm[t] = r_val**2
mkt_premium = (mkt_aligned - rf_daily).mean() * 252
mu_capm = pd.Series({
    t: rf_annual + betas_capm[t]*mkt_premium for t in PORTFOLIO_TICKERS}, name='E[r] CAPM')
print(f'Prime de marché (VT) : {mkt_premium:.4f}')
print(pd.concat([pd.Series(betas_capm,name='Beta'),pd.Series(r2_capm,name='R2')],axis=1).round(3))

In [ ]:
mu_compare = pd.DataFrame({'FF3':mu_ff3,'CAPM':mu_capm,'Historique':ann_ret.loc[PORTFOLIO_TICKERS]})
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(PORTFOLIO_TICKERS)); width = 0.27
ax.bar(x-width, mu_compare['FF3'],        width, label='FF3',        color='#d62728', alpha=0.85)
ax.bar(x,       mu_compare['CAPM'],       width, label='CAPM',       color='#1f77b4', alpha=0.85)
ax.bar(x+width, mu_compare['Historique'], width, label='Historique', color='#2ca02c', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(PORTFOLIO_TICKERS, rotation=45, ha='right')
ax.set_title('Rendements attendus : FF3 vs CAPM vs Historique', fontweight='bold')
ax.axhline(0, color='black', lw=0.8); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
cov = R.cov() * 252
n   = len(PORTFOLIO_TICKERS)
def portfolio_metrics(w, mu, cov, rf):
    ret=w@mu; vol=np.sqrt(w@cov@w); sr=(ret-rf)/vol
    return ret, vol, sr
def neg_sharpe(w, mu, cov, rf): return -portfolio_metrics(w,mu,cov,rf)[2]
constraints = [{'type':'eq','fun':lambda w: w.sum()-1}]
bounds = [(0.0,0.40)]*n; w0 = np.ones(n)/n
opts   = {'ftol':1e-12,'maxiter':2000}
res_ff3  = minimize(neg_sharpe, w0, args=(mu_ff3.values,cov.values,rf_annual),
                    method='SLSQP',bounds=bounds,constraints=constraints,options=opts)
w_ff3 = pd.Series(res_ff3.x, index=PORTFOLIO_TICKERS)
res_capm = minimize(neg_sharpe, w0, args=(mu_capm.values,cov.values,rf_annual),
                    method='SLSQP',bounds=bounds,constraints=constraints,options=opts)
w_capm = pd.Series(res_capm.x, index=PORTFOLIO_TICKERS)
w_ew   = pd.Series(np.ones(n)/n, index=PORTFOLIO_TICKERS)
def fmt_m(w,mu,label):
    r,v,s=portfolio_metrics(w.values,mu.values,cov.values,rf_annual)
    return pd.Series({'Rendement attendu':r,'Volatilité':v,'Sharpe':s},name=label)
summary=pd.concat([fmt_m(w_ff3,mu_ff3,'FF3'),fmt_m(w_capm,mu_capm,'CAPM'),fmt_m(w_ew,mu_ff3,'EW')],axis=1)
print('=== Portefeuilles optimaux (ex-ante) ===\n'); print(summary.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Poids optimaux — FF3 vs CAPM', fontsize=14, fontweight='bold')
for ax, (weights, model, color) in zip(axes,[
    (w_ff3,'FF3 Max Sharpe','#d62728'),(w_capm,'CAPM Max Sharpe','#1f77b4')]):
    w_plot = weights[weights>0.005].sort_values(ascending=True)
    w_plot.plot(kind='barh', ax=ax, color=color, alpha=0.85)
    ax.axvline(1/n, color='grey', linestyle='--', lw=1.2, label=f'EW ({1/n:.1%})')
    for i, val in enumerate(w_plot): ax.text(val+0.003,i,f'{val:.1%}',va='center',fontsize=9)
    ax.set_title(model); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax,(weights,model) in zip(axes,[(w_ff3,'FF3'),(w_capm,'CAPM')]):
    w_plot=weights[weights>0.005]
    ax.pie(w_plot,labels=w_plot.index,autopct='%1.1f%%',startangle=90,textprops={'fontsize':8})
    ax.set_title(model,fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
N_SIM=6000; sim_ret=np.zeros(N_SIM); sim_vol=np.zeros(N_SIM); sim_sr=np.zeros(N_SIM)
rng=np.random.default_rng(42)
for i in range(N_SIM):
    w=rng.random(n); w/=w.sum()
    r,v,s=portfolio_metrics(w,mu_ff3.values,cov.values,rf_annual)
    sim_ret[i]=r; sim_vol[i]=v; sim_sr[i]=s
r_ff3,v_ff3,s_ff3=portfolio_metrics(w_ff3.values,mu_ff3.values,cov.values,rf_annual)
r_capm,v_capm,s_capm=portfolio_metrics(w_capm.values,mu_capm.values,cov.values,rf_annual)
r_ew,v_ew,s_ew=portfolio_metrics(w_ew.values,mu_ff3.values,cov.values,rf_annual)
fig,ax=plt.subplots(figsize=(11,7))
sc=ax.scatter(sim_vol,sim_ret,c=sim_sr,cmap='viridis',alpha=0.35,s=12)
plt.colorbar(sc,ax=ax,label='Sharpe Ratio')
for t in PORTFOLIO_TICKERS:
    ax.scatter(ann_vol[t],ann_ret[t],s=40,color='grey',zorder=4,alpha=0.7)
    ax.annotate(t,(ann_vol[t],ann_ret[t]),fontsize=7,color='grey',textcoords='offset points',xytext=(4,2))
ax.scatter(v_ff3,r_ff3,s=250,color='red',zorder=6,marker='*',label=f'FF3 SR={s_ff3:.2f}')
ax.scatter(v_capm,r_capm,s=250,color='blue',zorder=6,marker='*',label=f'CAPM SR={s_capm:.2f}')
ax.scatter(v_ew,r_ew,s=150,color='green',zorder=6,marker='D',label=f'EW SR={s_ew:.2f}')
ax.set_xlabel('Volatilité'); ax.set_ylabel('Rendement attendu (FF3)')
ax.set_title('Frontière Efficiente — Monte Carlo',fontweight='bold',fontsize=13)
ax.legend(fontsize=10); plt.tight_layout(); plt.show()

## 5. Benchmark Construction

Benchmark par défaut : **60% VT** (actions mondiales) **+ 40% AGG** (obligations US).

In [ ]:
bench_w = pd.Series({'VT': 0.60, 'AGG': 0.40})
bench_prices = raw[['VT','AGG']].ffill().dropna()
bench_daily  = np.log(bench_prices / bench_prices.shift(1)).dropna()
bench_6040   = (bench_daily * bench_w).sum(axis=1).reindex(common_idx).dropna()
bench_6040.name = '60/40 Benchmark'

cum_b   = np.exp(bench_6040.cumsum()) - 1
ret_b   = bench_6040.mean() * 252
vol_b   = bench_6040.std()  * np.sqrt(252)
sr_b    = (ret_b - rf_annual) / vol_b
dd_b    = (np.exp(bench_6040.cumsum()) / np.exp(bench_6040.cumsum()).cummax() - 1).min()

print('=== Benchmark 60/40 (VT 60% + AGG 40%) ===')
print(f'Rendement annualisé  : {ret_b:.2%}')
print(f'Volatilité annualisée: {vol_b:.2%}')
print(f'Sharpe Ratio         : {sr_b:.3f}')
print(f'Max Drawdown         : {dd_b:.2%}')

fig, ax = plt.subplots(figsize=(12, 4))
cum_b.plot(ax=ax, color='black', lw=2)
ax.fill_between(cum_b.index, 0, cum_b, where=cum_b>=0, alpha=0.15, color='green')
ax.fill_between(cum_b.index, 0, cum_b, where=cum_b<0,  alpha=0.15, color='red')
ax.set_title('Performance cumulée — Benchmark 60/40', fontweight='bold')
ax.set_ylabel('Rendement cumulé'); ax.axhline(0, color='grey', lw=0.8)
plt.tight_layout(); plt.show()

## 6. Factor Regression — FF6 (FF5 + Momentum)

Régression sur le **sous-portefeuille US Equity** (VIOV, MTUM, VBR) avec 6 facteurs :

$$r_i - r_f = \alpha + \beta_{\text{mkt}}\cdot MKT + \beta_{\text{smb}}\cdot SMB + \beta_{\text{hml}}\cdot HML + \beta_{\text{rmw}}\cdot RMW + \beta_{\text{cma}}\cdot CMA + \beta_{\text{mom}}\cdot MOM + \varepsilon$$

In [ ]:
# Téléchargement FF5 quotidien
ff5_raw = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start=START)[0]
if not isinstance(ff5_raw.index, pd.DatetimeIndex):
    ff5_raw.index = pd.to_datetime(ff5_raw.index.astype(str), format='%Y%m%d')
ff5 = ff5_raw / 100

# Téléchargement Momentum quotidien
mom_raw = web.DataReader('F-F_Momentum_Factor_daily', 'famafrench', start=START)[0]
if not isinstance(mom_raw.index, pd.DatetimeIndex):
    mom_raw.index = pd.to_datetime(mom_raw.index.astype(str), format='%Y%m%d')
mom = mom_raw / 100

# Fusion FF6 = FF5 + Mom
FACTORS_FF6 = ['Mkt-RF','SMB','HML','RMW','CMA','Mom']
ff6_all = ff5[['Mkt-RF','SMB','HML','RMW','CMA']].join(mom[['Mom']], how='inner')

# Alignement avec le sous-portefeuille US Equity
us_common  = R[US_SUB].index.intersection(ff6_all.index)
R_us       = R.loc[us_common, US_SUB]
ff6_al     = ff6_all.loc[us_common]
rf_us      = F.loc[us_common, 'RF']

betas_ff6 = {}; alphas_ff6 = {}; r2_ff6 = {}
for t in US_SUB:
    excess = (R_us[t] - rf_us).values
    X = np.column_stack([np.ones(len(ff6_al)), ff6_al[FACTORS_FF6].values])
    coeffs,_,_,_ = np.linalg.lstsq(X, excess, rcond=None)
    y_pred = X @ coeffs
    ss_res = np.sum((excess - y_pred)**2)
    ss_tot = np.sum((excess - excess.mean())**2)
    alphas_ff6[t] = coeffs[0] * 252
    betas_ff6[t]  = dict(zip(FACTORS_FF6, coeffs[1:]))
    r2_ff6[t]     = 1 - ss_res/ss_tot

betas_ff6_df = pd.DataFrame(betas_ff6).T
print('=== FF6 — US Equity Sub-Portfolio ===\n')
print('Bêtas :'); print(betas_ff6_df.round(4))
print('\nAlphas annualisés :', {t:round(v,4) for t,v in alphas_ff6.items()})
print('R²              :', {t:round(v,4) for t,v in r2_ff6.items()})

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('FF6 — Bêtas par facteur (US Equity Sub-Portfolio)', fontsize=13, fontweight='bold')
f_colors = ['#d62728','#1f77b4','#2ca02c','#ff7f0e','#9467bd','#8c564b']
for ax,(factor,color) in zip(axes.flatten(), zip(FACTORS_FF6, f_colors)):
    betas_ff6_df[factor].plot(kind='bar', ax=ax, color=color, alpha=0.85)
    ax.axhline(0, color='black', lw=0.8); ax.set_title(f'β {factor}')
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

# Heatmap FF6
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(betas_ff6_df.values, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=2)
ax.set_xticks(range(6)); ax.set_xticklabels(FACTORS_FF6)
ax.set_yticks(range(len(US_SUB))); ax.set_yticklabels(US_SUB)
for i in range(len(US_SUB)):
    for j in range(6):
        ax.text(j, i, f'{betas_ff6_df.values[i,j]:.2f}', ha='center', va='center', fontsize=10, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.02)
ax.set_title('Heatmap bêtas FF6 — US Equity', fontweight='bold')
plt.tight_layout(); plt.show()

## 7. Backtest & Performance Analysis

In [ ]:
port_ff3  = (R * w_ff3.values).sum(axis=1)
port_capm = (R * w_capm.values).sum(axis=1)
port_ew   = (R * w_ew.values).sum(axis=1)
port_vt   = np.log(raw['VT'] / raw['VT'].shift(1)).dropna().loc[common_idx]
cum_ff3   = np.exp(port_ff3.cumsum())  - 1
cum_capm  = np.exp(port_capm.cumsum()) - 1
cum_ew    = np.exp(port_ew.cumsum())   - 1
cum_vt    = np.exp(port_vt.cumsum())   - 1
fig, ax = plt.subplots(figsize=(13, 6))
cum_ff3.plot(ax=ax,  label='FF3 Max Sharpe',  color='#d62728', lw=1.8)
cum_capm.plot(ax=ax, label='CAPM Max Sharpe', color='#1f77b4', lw=1.8)
cum_ew.plot(ax=ax,   label='Equal Weight',    color='#2ca02c', lw=1.5, linestyle='--')
cum_vt.plot(ax=ax,   label='VT Benchmark',    color='black',   lw=1.5, linestyle=':')
(np.exp(bench_6040.cumsum())-1).plot(ax=ax, label='60/40 Benchmark', color='orange', lw=1.5, linestyle='-.')
ax.set_title('Backtest — Performance cumulée', fontweight='bold', fontsize=13)
ax.set_ylabel('Rendement cumulé'); ax.axhline(0, color='grey', lw=0.8); ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

def perf_metrics(port_ret, rf_ann, label):
    ann_r=port_ret.mean()*252; ann_v=port_ret.std()*np.sqrt(252)
    sr=(ann_r-rf_ann)/ann_v
    cum=np.exp(port_ret.cumsum()); max_dd=(cum/cum.cummax()-1).min()
    calmar=ann_r/abs(max_dd) if max_dd!=0 else np.nan
    return pd.Series({'Rendement annualisé':ann_r,'Volatilité':ann_v,
                      'Sharpe Ratio':sr,'Max Drawdown':max_dd,'Calmar':calmar},name=label)

metrics = pd.concat([
    perf_metrics(port_ff3,  rf_annual,'FF3 Max Sharpe'),
    perf_metrics(port_capm, rf_annual,'CAPM Max Sharpe'),
    perf_metrics(port_ew,   rf_annual,'Equal Weight'),
    perf_metrics(port_vt,   rf_annual,'VT Benchmark'),
    perf_metrics(bench_6040,rf_annual,'60/40 Benchmark'),
], axis=1)
print('=== Métriques ex-post ===\n'); print(metrics.round(4))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Métriques de performance ex-post', fontsize=14, fontweight='bold')
for ax,(metric,color) in zip(axes.flatten(),[
    ('Rendement annualisé','#2ca02c'),('Volatilité','#d62728'),
    ('Sharpe Ratio','#1f77b4'),('Max Drawdown','#ff7f0e')]):
    vals=metrics.loc[metric]
    vals.plot(kind='bar',ax=ax,color=color,alpha=0.85)
    ax.set_title(metric); ax.set_xticklabels(metrics.columns,rotation=20,ha='right')
    ax.axhline(0,color='grey',lw=0.8)
plt.tight_layout(); plt.show()

In [ ]:
# --- Backtest avec rebalancement mensuel ---
def backtest_rebalanced(weights, returns_df):
    parts = []
    for _, grp in returns_df.groupby(pd.Grouper(freq='ME')):
        if len(grp): parts.append((grp * weights).sum(axis=1))
    return pd.concat(parts).sort_index()

port_ff3_reb  = backtest_rebalanced(w_ff3.values,  R)
port_capm_reb = backtest_rebalanced(w_capm.values, R)

fig, ax = plt.subplots(figsize=(13, 5))
(np.exp(port_ff3_reb.cumsum())-1).plot(ax=ax, label='FF3 — Rebalancé mensuel', color='#d62728', lw=1.8)
(np.exp(port_ff3.cumsum())-1).plot(ax=ax,     label='FF3 — Buy & Hold',        color='#d62728', lw=1.5, linestyle='--')
(np.exp(bench_6040.cumsum())-1).plot(ax=ax,   label='60/40 Benchmark',          color='black',   lw=1.5, linestyle=':')
ax.set_title('Rebalancé mensuel vs Buy & Hold — FF3', fontweight='bold')
ax.set_ylabel('Rendement cumulé'); ax.legend(); ax.axhline(0,color='grey',lw=0.8)
plt.tight_layout(); plt.show()

# --- Attribution de performance (contribution par actif) ---
contrib_ff3  = (R * w_ff3.values).mean()  * 252
contrib_capm = (R * w_capm.values).mean() * 252
contrib_df   = pd.DataFrame({'FF3':contrib_ff3,'CAPM':contrib_capm}).sort_values('FF3')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Attribution de Performance — Contribution annualisée par actif', fontweight='bold')
for ax,(col,color) in zip(axes,[('FF3','#d62728'),('CAPM','#1f77b4')]):
    c_sorted = contrib_df[col].sort_values()
    bar_c = ['#d62728' if v<0 else '#2ca02c' for v in c_sorted]
    c_sorted.plot(kind='barh', ax=ax, color=bar_c, alpha=0.85)
    ax.axvline(0,color='grey',lw=0.8); ax.set_title(col)
plt.tight_layout(); plt.show()

# --- Rolling Sharpe 252 jours ---
def rolling_sr(s, rf, w=252): return (s.rolling(w).mean()*252 - rf) / (s.rolling(w).std()*np.sqrt(252))
fig, ax = plt.subplots(figsize=(13, 4))
rolling_sr(port_ff3,  rf_annual).plot(ax=ax, label='FF3',           color='#d62728', lw=1.5)
rolling_sr(port_capm, rf_annual).plot(ax=ax, label='CAPM',          color='#1f77b4', lw=1.5)
rolling_sr(bench_6040,rf_annual).plot(ax=ax, label='60/40 Benchmark',color='black', lw=1.2, linestyle='--')
ax.axhline(0,color='grey',lw=0.8); ax.set_title('Sharpe Ratio Glissant (252j)',fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

## 8. Glide Path Simulation

Réduction systématique de l'exposition actions de **80% → 40%** sur les 10 dernières années du backtest.

In [ ]:
# Période : 10 dernières années du backtest
gp_start = R.index[-1] - pd.DateOffset(years=10)
gp_dates = R.index[R.index >= gp_start]
n_days   = len(gp_dates)

# Interpolation linéaire de l'allocation actions
eq_alloc   = np.linspace(0.80, 0.40, n_days)
bond_alloc = 1 - eq_alloc

# Poches equity et bonds dans PORTFOLIO_TICKERS
BOND_ASSETS   = ['AGG', 'BNDX']
EQUITY_ASSETS = [t for t in PORTFOLIO_TICKERS if t not in BOND_ASSETS]
w_eq_sub   = pd.Series({t: 1/len(EQUITY_ASSETS) for t in EQUITY_ASSETS})
w_bond_sub = pd.Series({t: 1/len(BOND_ASSETS)   for t in BOND_ASSETS})

R_gp = R.loc[gp_dates]
ret_eq   = (R_gp[EQUITY_ASSETS] * w_eq_sub).sum(axis=1).values
ret_bond = (R_gp[BOND_ASSETS]   * w_bond_sub).sum(axis=1).values
gp_daily = pd.Series(eq_alloc*ret_eq + bond_alloc*ret_bond, index=gp_dates, name='Glide Path')

# Visualisation
fig, axes = plt.subplots(2, 1, figsize=(13, 8), gridspec_kw={'height_ratios':[1,2]})
axes[0].fill_between(gp_dates, eq_alloc,   1, alpha=0.35, color='steelblue', label='Obligations')
axes[0].fill_between(gp_dates, 0, eq_alloc, alpha=0.45, color='#d62728',    label='Actions')
axes[0].set_title('Évolution de l\'allocation — Glide Path', fontweight='bold')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))
axes[0].legend(loc='upper right')

(np.exp(gp_daily.cumsum())-1).plot(ax=axes[1], label='Glide Path', color='steelblue', lw=2)
(np.exp(port_ff3.loc[gp_dates].cumsum())-1).plot(ax=axes[1], label='FF3 (fixe)', color='#d62728', lw=1.8, linestyle='--')
(np.exp(bench_6040.loc[gp_dates].cumsum())-1).plot(ax=axes[1], label='60/40', color='black', lw=1.5, linestyle=':')
axes[1].set_title('Performance cumulée — 10 dernières années', fontweight='bold')
axes[1].axhline(0,color='grey',lw=0.8); axes[1].legend()
plt.tight_layout(); plt.show()

gp_ann_ret = gp_daily.mean()*252; gp_ann_vol = gp_daily.std()*np.sqrt(252)
gp_cum = np.exp(gp_daily.cumsum()); gp_dd = (gp_cum/gp_cum.cummax()-1).min()
print(f'Glide Path — Rendement : {gp_ann_ret:.2%} | Vol : {gp_ann_vol:.2%} | '
      f'Sharpe : {(gp_ann_ret-rf_annual)/gp_ann_vol:.3f} | MaxDD : {gp_dd:.2%}')

## 9. Asynchronous Trading Adjustment

Les marchés US, Europe et Asie ne ferment pas simultanément. Les corrélations sur données **journalières** sous-estiment les vraies dépendances cross-asset. Solution : utiliser des **rendements hebdomadaires** (vendredi au vendredi).

In [ ]:
# Resample en hebdomadaire (clôture vendredi)
weekly_prices  = prices.resample('W-FRI').last()
weekly_returns = np.log(weekly_prices / weekly_prices.shift(1)).dropna()

daily_corr  = returns.corr()
weekly_corr = weekly_returns.corr()
diff_corr   = weekly_corr - daily_corr

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Ajustement Asynchrone — Corrélations Journalières vs Hebdomadaires', fontsize=12, fontweight='bold')
for ax,(mat,title,vmin,vmax,cmap) in zip(axes,[
    (daily_corr, 'Journalières',      -1,   1, 'RdYlGn'),
    (weekly_corr,'Hebdomadaires',     -1,   1, 'RdYlGn'),
    (diff_corr,  'Différence (H−J)', -0.3, 0.3,'RdBu_r'),
]):
    im = ax.imshow(mat.values, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    ax.set_xticks(range(len(PORTFOLIO_TICKERS)))
    ax.set_yticks(range(len(PORTFOLIO_TICKERS)))
    ax.set_xticklabels(mat.columns, rotation=90, fontsize=7)
    ax.set_yticklabels(mat.index, fontsize=7)
    ax.set_title(title, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.03)
plt.tight_layout(); plt.show()

# Régression FF6 sur rendements hebdomadaires — US Equity
ff6_weekly = ff6_all.resample('W-FRI').sum()   # somme des facteurs quotidiens sur la semaine
rf_weekly  = F['RF'].resample('W-FRI').sum()
w_idx      = weekly_returns.index.intersection(ff6_weekly.index).intersection(rf_weekly.index)
wR = weekly_returns.loc[w_idx, US_SUB]
wF = ff6_weekly.loc[w_idx, FACTORS_FF6]
wRF= rf_weekly.loc[w_idx]

betas_ff6_wk = {}
for t in US_SUB:
    excess = (wR[t] - wRF).values
    X = np.column_stack([np.ones(len(wF)), wF.values])
    coeffs,_,_,_ = np.linalg.lstsq(X, excess, rcond=None)
    betas_ff6_wk[t] = dict(zip(FACTORS_FF6, coeffs[1:]))

betas_ff6_wk_df = pd.DataFrame(betas_ff6_wk).T
print('=== Bêtas FF6 — Journalier vs Hebdomadaire (US Equity) ===\n')
print('Journalier :'); print(betas_ff6_df.round(4))
print('\nHebdomadaire :'); print(betas_ff6_wk_df.round(4))

# Impact sur la matrice de corrélation — paires cross-régions
cross_pairs = [('VIOV','MC.PA'),('MTUM','ASML'),('VBR','NESN.SW'),('VEA','VWO')]
print('\n=== Impact sur corrélations cross-régions ===')
print(f'{"Paire":<20} {"Journalière":>14} {"Hebdomadaire":>14} {"Δ":>8}')
for a,b in cross_pairs:
    if a in daily_corr.columns and b in daily_corr.columns:
        d=daily_corr.loc[a,b]; w=weekly_corr.loc[a,b]
        print(f'{a+" / "+b:<20} {d:>14.3f} {w:>14.3f} {w-d:>8.3f}')

In [ ]:
from IPython.display import Markdown, display

sr_ff3_ex  = metrics.loc['Sharpe Ratio', 'FF3 Max Sharpe']
sr_capm_ex = metrics.loc['Sharpe Ratio', 'CAPM Max Sharpe']
sr_vt      = metrics.loc['Sharpe Ratio', 'VT Benchmark']
ret_ff3_ex = metrics.loc['Rendement annualisé', 'FF3 Max Sharpe']
ret_capm_ex= metrics.loc['Rendement annualisé', 'CAPM Max Sharpe']
ret_vt_ex  = metrics.loc['Rendement annualisé', 'VT Benchmark']
dd_ff3     = metrics.loc['Max Drawdown', 'FF3 Max Sharpe']
dd_capm    = metrics.loc['Max Drawdown', 'CAPM Max Sharpe']
dd_vt      = metrics.loc['Max Drawdown', 'VT Benchmark']
best       = 'FF3' if sr_ff3_ex >= sr_capm_ex else 'CAPM'
best_sr    = max(sr_ff3_ex, sr_capm_ex)
other      = 'CAPM' if best=='FF3' else 'FF3'
other_sr   = min(sr_ff3_ex, sr_capm_ex)
top_ff3    = w_ff3.idxmax(); top_capm = w_capm.idxmax()
w_ff3_top  = w_ff3.max();    w_capm_top = w_capm.max()
only_ff3   = [t for t in PORTFOLIO_TICKERS if w_ff3[t]>0.01 and w_capm[t]<=0.01]
only_capm  = [t for t in PORTFOLIO_TICKERS if w_capm[t]>0.01 and w_ff3[t]<=0.01]
common_w   = [t for t in PORTFOLIO_TICKERS if w_ff3[t]>0.01 and w_capm[t]>0.01]
alpha_ff3  = ret_ff3_ex  - metrics.loc['Rendement annualisé','60/40 Benchmark']
alpha_capm = ret_capm_ex - metrics.loc['Rendement annualisé','60/40 Benchmark']

txt = f'''
---
## Conclusion Intermédiaire — Comparaison FF3 vs CAPM

### Divergence des poids
| Critère | FF3 | CAPM |
|---|---|---|
| Actif dominant | **{top_ff3}** ({w_ff3_top:.1%}) | **{top_capm}** ({w_capm_top:.1%}) |
| Actifs exclusifs | {', '.join(only_ff3) if only_ff3 else '—'} | {', '.join(only_capm) if only_capm else '—'} |
| Actifs communs (>1%) | {', '.join(common_w) if common_w else '—'} | ← idem |

Le FF3 concentre sur les actifs à fort bêta **SMB/HML** (primes Small Cap & Value).
Le CAPM alloue uniquement sur le bêta marché, ignorant ces tilts factoriels.

### Performance ex-post vs Benchmark 60/40
| Métrique | FF3 | CAPM | 60/40 Benchmark |
|---|---|---|---|
| Rendement annualisé | {ret_ff3_ex:.2%} | {ret_capm_ex:.2%} | {metrics.loc["Rendement annualisé","60/40 Benchmark"]:.2%} |
| Sharpe Ratio | **{sr_ff3_ex:.3f}** | **{sr_capm_ex:.3f}** | {metrics.loc["Sharpe Ratio","60/40 Benchmark"]:.3f} |
| Max Drawdown | {dd_ff3:.2%} | {dd_capm:.2%} | {metrics.loc["Max Drawdown","60/40 Benchmark"]:.2%} |
| Alpha vs 60/40 | {alpha_ff3:+.2%} | {alpha_capm:+.2%} | — |

### Modèle recommandé : {best}
Sharpe ex-post **{best_sr:.3f}** vs {other_sr:.3f} pour le {other}.
Le {best} est désigné comme portefeuille de référence pour Sofia.
Recommandation : rebalancement trimestriel, revue annuelle des facteurs.
'''
display(Markdown(txt))